In [1]:
import os
os.chdir("../")
%pwd

'c:\\Users\\12345\\OneDrive\\Desktop\\experiments & internship\\ML-Projects'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [12]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )

        return model_trainer_config

In [13]:
import pandas as pd
import os
from mlProject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [16]:
class ModelTrainer:
    
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        try:
            # Load training data
            train_data = pd.read_csv(self.config.train_data_path)
            test_data = pd.read_csv(self.config.test_data_path)

            X_train = train_data.drop([self.config.target_column], axis=1)
            y_train = train_data[self.config.target_column]
            X_test = test_data.drop([self.config.target_column], axis=1)
            y_test = test_data[self.config.target_column]

            # Initialize and train the model
            model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
            model.fit(X_train, y_train)

            # Save the trained model
            model_path = os.path.join(self.config.root_dir, self.config.model_name)
            joblib.dump(model, model_path)
            logger.info(f"Model saved at {model_path}")

        except Exception as e:
            logger.error(f"Error occurred while training the model: {e}")

In [17]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    raise e

[2026-09-21 01:10:01,502]: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-21 01:10:01,504]: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-21 01:10:01,506]: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-21 01:10:01,508]: INFO: common: Directory created at: artifacts]
[2026-09-21 01:10:01,510]: INFO: common: Directory created at: artifacts/model_trainer]
[2026-09-21 01:10:01,530]: INFO: 3063190505: Model saved at artifacts/model_trainer\model.joblib]
